## 05 — Gold Layer: Star Schema Materialisation

Reads from the silver table (`students_data.team4-chicago.food_inspections_silver`) produced by **03-Silver-Transformations** and materialises the star schema defined in **04-Star-Schema-Definition** as real Delta tables.

### Tables Created
| Table | Type | Grain | Key Logic |
| --- | --- | --- | --- |
| `dim_date` | Dimension | One row per calendar date | `date_key` = YYYYMMDD integer |
| `dim_business` | Dimension | One row per unique business profile | Surrogate key on (License, DBA_Name, AKA_Name, Facility_Type, Risk) |
| `dim_location` | Dimension | One row per unique address | Surrogate key on (Address, City, State, Zip) |
| `dim_inspection_type` | Dimension | One row per inspection type | Surrogate key on Inspection_Type |
| `dim_violation_type` | Dimension | One row per violation code | Surrogate key on code; MAX(description) picks best variant |
| `fact_violation` | Fact | One row per violation event | Surrogate `violation_id`, FKs to all dimensions |

### Pipeline Context
| Layer | Notebook | Output |
| --- | --- | --- |
| Bronze | `00-Input-Data` | `food_inspections_bronze` (307,282 rows) |
| Bronze | `01-Data-Profiling` | Data quality fixes applied in-memory |
| Bronze | `02-Schema-Enforcement` | NOT NULL constraints on bronze table |
| Silver | `03-Silver-Transformations` | `food_inspections_silver` (inspection grain, violations as struct array) |
| Gold | **This notebook** | **5 dimensions + 1 fact table** |

In [0]:
-- Dimension: Date — one row per unique calendar date
-- date_key is a compact YYYYMMDD integer for efficient joins and partitioning

CREATE OR REPLACE TABLE students_data.`team4-chicago`.dim_date AS
SELECT
  CAST(DATE_FORMAT(full_date, 'yyyyMMdd') AS INT)  AS date_key,
  full_date,
  YEAR(full_date)                                  AS year,
  MONTH(full_date)                                 AS month,
  DAY(full_date)                                   AS day,
  DATE_FORMAT(full_date, 'EEEE')                   AS day_of_week,
  QUARTER(full_date)                               AS quarter
FROM (
  SELECT DISTINCT Inspection_Date AS full_date
  FROM   students_data.`team4-chicago`.food_inspections_silver
  WHERE  Inspection_Date IS NOT NULL
)
ORDER BY full_date;

-- Preview the created table
SELECT * FROM students_data.`team4-chicago`.dim_date LIMIT 20;

In [0]:
-- Dimension: Business — one row per unique business profile
-- Keyed on the full natural key (License, DBA_Name, AKA_Name, Facility_Type, Risk)

CREATE OR REPLACE TABLE students_data.`team4-chicago`.dim_business AS
SELECT
  ROW_NUMBER() OVER (ORDER BY License, DBA_Name, Facility_Type, Risk)  AS business_key,
  License         AS license_number,
  DBA_Name        AS dba_name,
  AKA_Name        AS aka_name,
  Facility_Type   AS facility_type,
  Risk            AS risk
FROM (
  SELECT DISTINCT License, DBA_Name, AKA_Name, Facility_Type, Risk
  FROM   students_data.`team4-chicago`.food_inspections_silver
)
ORDER BY business_key;

-- Preview the created table
SELECT * FROM students_data.`team4-chicago`.dim_business LIMIT 20;

In [0]:
-- Dimension: Location — one row per unique address
-- Lat/lon resolved via MAX (naturally ignores nulls) per address group

CREATE OR REPLACE TABLE students_data.`team4-chicago`.dim_location AS
SELECT
  ROW_NUMBER() OVER (ORDER BY Address, City, State, Zip)  AS location_key,
  Address    AS address,
  City       AS city,
  State      AS state,
  Zip        AS zip,
  latitude,
  longitude
FROM (
  SELECT
    Address,
    City,
    State,
    Zip,
    MAX(Latitude)  AS latitude,
    MAX(Longitude) AS longitude
  FROM   students_data.`team4-chicago`.food_inspections_silver
  GROUP BY Address, City, State, Zip
)
ORDER BY location_key;

-- Preview the created table
SELECT * FROM students_data.`team4-chicago`.dim_location LIMIT 20;

In [0]:
-- Dimension: Inspection Type — one row per unique inspection type

CREATE OR REPLACE TABLE students_data.`team4-chicago`.dim_inspection_type AS
SELECT
  ROW_NUMBER() OVER (ORDER BY Inspection_Type)  AS inspection_type_key,
  Inspection_Type                                AS inspection_type
FROM (
  SELECT DISTINCT Inspection_Type
  FROM   students_data.`team4-chicago`.food_inspections_silver
  WHERE  Inspection_Type IS NOT NULL
)
ORDER BY inspection_type_key;

-- Preview the created table
SELECT * FROM students_data.`team4-chicago`.dim_inspection_type LIMIT 20;

In [0]:
-- Dimension: Violation Type — one row per unique violation code
-- Uses GROUP BY code with MAX(description) to:
--   1. Collapse multiple description variants per code into one row
--   2. Ignore nulls/empty strings (MAX skips nulls)
-- Note: Chicago updated violation descriptions over the years, so the same code
-- can have different description text. MAX picks the lexicographically largest
-- (typically the most descriptive / latest version).

CREATE OR REPLACE TABLE students_data.`team4-chicago`.dim_violation_type AS
SELECT
  ROW_NUMBER() OVER (ORDER BY violation_code)  AS violation_type_key,
  violation_code,
  violation_description
FROM (
  SELECT
    v_code AS violation_code,
    MAX(v_description) AS violation_description
  FROM   students_data.`team4-chicago`.food_inspections_silver
  LATERAL VIEW INLINE(violations_parsed) v AS v_code, v_description, v_comment
  WHERE  v_code IS NOT NULL
  GROUP BY v_code
)
ORDER BY violation_type_key;

-- Preview the created table
SELECT * FROM students_data.`team4-chicago`.dim_violation_type LIMIT 20;

In [0]:
-- Fact: Violation — one row per violation event, linked to all dimensions
-- Explodes the struct array and joins to each dimension for surrogate keys

CREATE OR REPLACE TABLE students_data.`team4-chicago`.fact_violation AS
WITH exploded AS (
  SELECT
    s.Inspection_ID,
    s.License,
    s.DBA_Name,
    s.AKA_Name,
    s.Facility_Type,
    s.Risk,
    s.Address,
    s.City,
    s.State,
    s.Zip,
    s.Inspection_Date,
    s.Inspection_Type,
    s.Results,
    v_code        AS violation_code,
    v_comment     AS violation_comment
  FROM students_data.`team4-chicago`.food_inspections_silver s
  LATERAL VIEW INLINE(violations_parsed) v AS v_code, v_description, v_comment
)
SELECT
  ROW_NUMBER() OVER (ORDER BY e.Inspection_ID, e.violation_code)  AS violation_id,
  e.Inspection_ID        AS inspection_id,
  b.business_key,
  l.location_key,
  d.date_key,
  it.inspection_type_key,
  e.Results              AS result,
  vt.violation_type_key,
  e.violation_comment
FROM exploded e
JOIN students_data.`team4-chicago`.dim_business b
  ON  e.License       <=> b.license_number
  AND e.DBA_Name      =   b.dba_name
  AND e.AKA_Name      <=> b.aka_name
  AND e.Facility_Type <=> b.facility_type
  AND e.Risk          <=> b.risk
JOIN students_data.`team4-chicago`.dim_location l
  ON  e.Address = l.address
  AND e.City    = l.city
  AND e.State   = l.state
  AND e.Zip     = l.zip
JOIN students_data.`team4-chicago`.dim_date d
  ON  e.Inspection_Date = d.full_date
JOIN students_data.`team4-chicago`.dim_inspection_type it
  ON  e.Inspection_Type = it.inspection_type
JOIN students_data.`team4-chicago`.dim_violation_type vt
  ON  e.violation_code = vt.violation_code;

-- Preview the created table
SELECT * FROM students_data.`team4-chicago`.fact_violation LIMIT 20;

In [0]:
-- Row counts for all gold tables
SELECT 'dim_date'            AS table_name, COUNT(*) AS row_count FROM students_data.`team4-chicago`.dim_date
UNION ALL
SELECT 'dim_business',        COUNT(*) FROM students_data.`team4-chicago`.dim_business
UNION ALL
SELECT 'dim_location',        COUNT(*) FROM students_data.`team4-chicago`.dim_location
UNION ALL
SELECT 'dim_inspection_type', COUNT(*) FROM students_data.`team4-chicago`.dim_inspection_type
UNION ALL
SELECT 'dim_violation_type',  COUNT(*) FROM students_data.`team4-chicago`.dim_violation_type
UNION ALL
SELECT 'fact_violation',      COUNT(*) FROM students_data.`team4-chicago`.fact_violation
ORDER BY table_name;